In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import numpy as np
import random
from tqdm import tqdm
from datasets import load_dataset
from transformers import pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

dataset = load_dataset("squad", split="validation[:700]")

TRAIN_SIZE = 500
TEST_SIZE = 200

full_data = dataset.select(range(700))

train_data = full_data.select(range(TRAIN_SIZE))
test_data = full_data.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))

In [4]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# optional (if you already have NLI model)
nli_pipeline = pipeline("text-classification", model="facebook/bart-large-mnli")
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# from transformers import AutoModelForSequenceClassification, AutoTokenizer 
# # BGE reranker uses a different interface than CrossEncoder
# bge_reranker = CrossEncoder("BAAI/bge-reranker-large")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [8]:
def normalize(x):
    x = np.array(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def get_rank(scores, labels):
    sorted_idx = np.argsort(scores)[::-1]
    ranks = np.where(labels[sorted_idx] == 1)[0]
    return ranks[0] + 1 if len(ranks) > 0 else len(scores)

def compute_metrics(ranks):
    ranks = np.array(ranks)
    return {
        "R@1": np.mean(ranks <= 1),
        "R@3": np.mean(ranks <= 3),
        "R@5": np.mean(ranks <= 5),
        "MRR": np.mean(1 / ranks),
        "Avg Rank": np.mean(ranks)
    }

# =========================
# CHUNKING (FIXED)
# =========================
def chunk_text(text, max_words=50, overlap=15):
    words = text.split()
    chunks = []
    step = max_words - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i+max_words])
        if len(chunk.strip()) > 0:
            chunks.append(chunk)

    return chunks

# =========================
# SCORING
# =========================
def get_chunk_embeddings(chunks):
    return bi_encoder.encode(chunks, convert_to_numpy=True)

def get_similarity_scores(q, chunks):
    q_emb = bi_encoder.encode([q], convert_to_numpy=True)[0]
    c_emb = get_chunk_embeddings(chunks)

    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_emb = c_emb / (np.linalg.norm(c_emb, axis=1, keepdims=True) + 1e-8)

    return c_emb @ q_emb

def get_cross_scores(q, chunks):
    pairs = [[q, c] for c in chunks]
    return cross_encoder.predict(pairs)

# =========================
# QA + NLI (FIXED)
# =========================
def get_evidence_scores(query, chunks):

    scores = []

    for chunk in chunks:
        try:
            qa_out = qa_pipeline(question=query, context=chunk)
            answer = qa_out["answer"]
            conf = qa_out["score"]

            if conf < 0.3 or not answer.strip():
                scores.append(0.0)
                continue

            hypothesis = f"The answer is {answer}."

            nli_out = nli_pipeline(
                f"{chunk} </s> {hypothesis}",
                top_k=None
            )

            entail = 0.0
            for item in nli_out:
                if "ENTAIL" in item["label"].upper():
                    entail = item["score"]

            score = entail * conf

        except:
            score = 0.0

        scores.append(score)

    return np.array(scores)

# =========================
# RAG DATASET (FIXED)
# =========================
def build_rag_dataset(data, num_distractors):

    rag_data = []
    all_contexts = [d["context"] for d in data]

    for sample in data:

        q = sample["question"]
        context = sample["context"]
        answer = sample["answers"]["text"][0]

        chunks = chunk_text(context)

        labels = [1 if answer in c else 0 for c in chunks]

        if sum(labels) == 0:
            continue

        distractors = random.sample(all_contexts, num_distractors)
        distractor_chunks = []

        for d in distractors:
            distractor_chunks.extend(chunk_text(d)[:1])

        chunks = chunks + distractor_chunks
        labels = labels + [0] * len(distractor_chunks)

        rag_data.append({
            "query": q,
            "chunks": chunks,
            "labels": labels
        })

    return rag_data

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def get_nli_scores_batch(chunks, hypotheses, batch_size=16):

    inputs = [f"{c} </s> {h}" for c, h in zip(chunks, hypotheses)]

    outputs = nli_pipeline(
        inputs,
        batch_size=batch_size,
        truncation=True,
        top_k=None   # IMPORTANT
    )

    scores = []

    for out in outputs:

        entail = 0.0

        # Case 1: list of dicts
        if isinstance(out, list):
            for item in out:
                if isinstance(item, dict) and "ENTAIL" in item.get("label", "").upper():
                    entail = item.get("score", 0.0)

        # Case 2: single dict
        elif isinstance(out, dict):
            if "ENTAIL" in out.get("label", "").upper():
                entail = out.get("score", 0.0)

        # Case 3: string label (rare but happens)
        elif isinstance(out, str):
            if "ENTAIL" in out.upper():
                entail = 1.0   # fallback (no score available)

        scores.append(entail)

    return np.array(scores)

def get_evidence_scores_fast(query, chunks):
    answers = []
    confs = []
    for chunk in chunks:
        try:
            qa_out = qa_pipeline(question=query, context=chunk)
            answer = qa_out["answer"]
            conf = qa_out["score"]

            if conf < 0.3 or not answer.strip():
                answers.append("")
                confs.append(0.0)
            else:
                answers.append(answer)
                confs.append(conf)
        except:
            answers.append("")
            confs.append(0.0)
            
    hypotheses = [
        f"The answer is {a}." if a != "" else ""
        for a in answers
    ]

    # --- Step 3: batch NLI ONLY for valid answers ---
    valid_idx = [i for i, a in enumerate(answers) if a != ""]

    nli_scores = np.zeros(len(chunks))

    if len(valid_idx) > 0:

        valid_chunks = [chunks[i] for i in valid_idx]
        valid_hyp = [hypotheses[i] for i in valid_idx]

        entail_scores = get_nli_scores_batch(valid_chunks, valid_hyp)

        for idx, score in zip(valid_idx, entail_scores):
            nli_scores[idx] = score * confs[idx]

    return nli_scores



def physarum_v5(sim_matrix, ce_scores, nli_scores,
                k=10, alpha=10.0, tau=0.5,
                lam_pos=0.2, lam_neg=0.15):

    N = len(ce_scores)

    # --- kNN graph ---
    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1

    knn_mask = np.maximum(knn_mask, knn_mask.T)
    sim_sparse = sim_matrix * knn_mask

    # --- gating ---
    gate = 1 / (1 + np.exp(-alpha * (ce_scores - tau)))

    P_pos = np.zeros(N)
    P_neg = np.zeros(N)

    for i in range(N):
        for j in range(N):

            if i == j:
                continue

            # -------- positive propagation --------
            if ce_scores[j] > ce_scores[i] + 0.05:
                P_pos[i] += sim_sparse[i, j] * ce_scores[j] * gate[i]

            # -------- negative propagation (FIXED) --------
            if nli_scores[j] < 0.2:
                P_neg[i] += sim_sparse[i, j] * (0.5 - nli_scores[j]) * gate[i]

    return ce_scores + lam_pos * P_pos - lam_neg * P_neg
    
def get_physarum_scores_v5(q, chunks, sim, ce, nli):

    chunk_embs = get_chunk_embeddings(chunks)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-8)

    sim_matrix = chunk_embs @ chunk_embs.T

    # inject query relevance
    sim_matrix = sim_matrix * sim[None, :]

    return physarum_v5(sim_matrix, ce, nli)



# =========================
# CONFIG
# =========================
SEEDS = [42,199,69]
DISTRACTORS = [30,50]

TRAIN_SIZE = 500
TEST_SIZE = 200

# =========================
# SPLIT DATA
# =========================
full_data = dataset.select(range(700))

train_data = full_data.select(range(TRAIN_SIZE))
test_data = full_data.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))


# =========================
# EXPERIMENT FUNCTION
# =========================
def run_single_experiment(train_data, test_data, num_distractors, seed):

    random.seed(seed)
    np.random.seed(seed)

    # ---- Build datasets separately (NO leakage) ----
    train_rag = build_rag_dataset(train_data, num_distractors)
    test_rag = build_rag_dataset(test_data, num_distractors)

    # ---------------------
    # TRAIN FEATURES
    # ---------------------
    feature_sets = {
        "CE+NLI": [],
        "CE+NLI+Sim": [],
        "CE+Physarum": [],
        "NLI+Physarum": [],
        "CE+NLI+Physarum": [],
        "CE+NLI+Physarum+Sim": []
    }

    y_train = []

    for sample in tqdm(train_rag, desc=f"Train (d={num_distractors}, seed={seed})"):

        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        # =========================
        # FAST NLI (with CE filtering)
        # =========================
        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))

        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)

            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)

        # =========================
        # PHYSARUM v5
        # =========================
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        # =========================
        # FEATURES
        # =========================
        for i in range(len(chunks)):
            feature_sets["CE+NLI"].append([ce[i], nli[i]])
            feature_sets["CE+NLI+Sim"].append([ce[i], nli[i], sim[i]])
            feature_sets["CE+Physarum"].append([ce[i], phy[i]])
            feature_sets["NLI+Physarum"].append([nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum"].append([ce[i], nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum+Sim"].append([ce[i], nli[i], phy[i], sim[i]])

            y_train.append(labels[i])

    y_train = np.array(y_train)

    # ---------------------
    # TRAIN MODELS
    # ---------------------
    models = {}
    weights = {}

    for name, feats in feature_sets.items():
        X = np.array(feats)

        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        model.fit(X, y_train)

        models[name] = model
        weights[name] = model.coef_[0]

    # ---------------------
    # EVALUATION
    # ---------------------
    results = {
        "Sim": [], "CE": [], "NLI": [], "Physarum": [],
        "CE+NLI": [], "CE+NLI+Sim": [],
        "CE+Physarum": [], "NLI+Physarum": [],
        "CE+NLI+Physarum": [], "CE+NLI+Physarum+Sim": []
    }

    for sample in tqdm(test_rag, desc=f"Test (d={num_distractors}, seed={seed})"):

        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        # =========================
        # FAST NLI (with CE filtering)
        # =========================
        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))

        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)

            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)

        # =========================
        # PHYSARUM v5
        # =========================
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        # =========================
        # BASE RESULTS
        # =========================
        results["Sim"].append(get_rank(sim, labels))
        results["CE"].append(get_rank(ce, labels))
        results["NLI"].append(get_rank(nli, labels))
        results["Physarum"].append(get_rank(phy, labels))

        # =========================
        # LEARNED
        # =========================
        feat_map = {
            "CE+NLI": np.stack([ce, nli], axis=1),
            "CE+NLI+Sim": np.stack([ce, nli, sim], axis=1),
            "CE+Physarum": np.stack([ce, phy], axis=1),
            "NLI+Physarum": np.stack([nli, phy], axis=1),
            "CE+NLI+Physarum": np.stack([ce, nli, phy], axis=1),
            "CE+NLI+Physarum+Sim": np.stack([ce, nli, phy, sim], axis=1),
        }

        for name, feat in feat_map.items():
            scores = models[name].predict_proba(feat)[:, 1]
            results[name].append(get_rank(scores, labels))

    # ---------------------
    # METRICS
    # ---------------------
    metrics = {k: compute_metrics(v) for k, v in results.items()}

    return metrics, weights

In [9]:
all_results = {}
all_weights = {}

for d in DISTRACTORS:
    print(f"\n==================== DISTRACTORS = {d} ====================")

    seed_metrics = []
    seed_weights = []

    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")

        metrics, weights = run_single_experiment(train_data, test_data, d, seed)

        seed_metrics.append(metrics)
        seed_weights.append(weights)

        # print metrics
        print("\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
        print("|--------|-----|-----|-----|-----|----------|")

        for k, v in metrics.items():
            print(f"| {k} | {v['R@1']:.3f} | {v['R@3']:.3f} | {v['R@5']:.3f} | {v['MRR']:.3f} | {v['Avg Rank']:.2f} |")

        print("\nWeights:")
        for name, w in weights.items():
            print(name, np.round(w, 3))

    # =========================
    # AVERAGE OVER SEEDS
    # =========================
    print(f"\n=== AVERAGE RESULTS (d={d}) ===")

    avg_metrics = {}

    for key in seed_metrics[0].keys():
        avg_metrics[key] = {}

        for m in ["R@1","R@3","R@5","MRR","Avg Rank"]:
            avg_metrics[key][m] = np.mean([s[key][m] for s in seed_metrics])

    print("\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
    print("|--------|-----|-----|-----|-----|----------|")

    for k, v in avg_metrics.items():
        print(f"| {k} | {v['R@1']:.3f} | {v['R@3']:.3f} | {v['R@5']:.3f} | {v['MRR']:.3f} | {v['Avg Rank']:.2f} |")

    # =========================
    # AVERAGE WEIGHTS
    # =========================
    print("\n=== AVERAGE WEIGHTS ===")

    for name in seed_weights[0].keys():
        w_avg = np.mean([sw[name] for sw in seed_weights], axis=0)
        print(name, np.round(w_avg, 3))

    all_results[d] = avg_metrics
    all_weights[d] = seed_weights


==================== DISTRACTORS = 30 ====================

--- Seed 42 ---


Test (d=30, seed=42): 100%|██████████| 200/200 [00:48<00:00,  4.11it/s]



| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.430 | 0.800 | 0.880 | 0.620 | 3.19 |
| CE | 0.505 | 0.875 | 0.960 | 0.693 | 2.08 |
| NLI | 0.465 | 0.750 | 0.810 | 0.617 | 5.62 |
| Physarum | 0.560 | 0.865 | 0.950 | 0.718 | 2.13 |
| CE+NLI | 0.580 | 0.925 | 0.980 | 0.746 | 1.79 |
| CE+NLI+Sim | 0.580 | 0.930 | 0.975 | 0.747 | 1.81 |
| CE+Physarum | 0.575 | 0.910 | 0.975 | 0.741 | 1.85 |
| NLI+Physarum | 0.620 | 0.910 | 0.960 | 0.766 | 1.79 |
| CE+NLI+Physarum | 0.650 | 0.955 | 0.985 | 0.791 | 1.61 |
| CE+NLI+Physarum+Sim | 0.635 | 0.950 | 0.985 | 0.782 | 1.64 |

Weights:
CE+NLI [5.246 1.867]
CE+NLI+Sim [3.893 1.873 2.256]
CE+Physarum [5.82  1.023]
NLI+Physarum [3.14  5.985]
CE+NLI+Physarum [4.494 1.896 1.326]
CE+NLI+Physarum+Sim [3.039 1.9   1.424 2.371]

--- Seed 199 ---


Test (d=30, seed=199): 100%|██████████| 200/200 [00:50<00:00,  3.95it/s]



| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.380 | 0.790 | 0.875 | 0.587 | 3.20 |
| CE | 0.465 | 0.880 | 0.965 | 0.665 | 2.16 |
| NLI | 0.390 | 0.735 | 0.825 | 0.570 | 5.75 |
| Physarum | 0.520 | 0.830 | 0.940 | 0.686 | 2.35 |
| CE+NLI | 0.505 | 0.920 | 0.985 | 0.697 | 1.93 |
| CE+NLI+Sim | 0.510 | 0.900 | 0.980 | 0.696 | 1.97 |
| CE+Physarum | 0.565 | 0.875 | 0.975 | 0.724 | 1.95 |
| NLI+Physarum | 0.570 | 0.890 | 0.970 | 0.732 | 1.89 |
| CE+NLI+Physarum | 0.605 | 0.945 | 0.990 | 0.761 | 1.70 |
| CE+NLI+Physarum+Sim | 0.610 | 0.925 | 0.990 | 0.762 | 1.72 |

Weights:
CE+NLI [5.208 1.911]
CE+NLI+Sim [3.46  1.916 3.003]
CE+Physarum [5.689 1.254]
NLI+Physarum [3.152 6.072]
CE+NLI+Physarum [4.277 1.967 1.635]
CE+NLI+Physarum+Sim [2.382 1.975 1.77  3.216]

--- Seed 69 ---


...


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.355 | 0.620 | 0.795 | 0.530 | 4.58 |
| CE | 0.485 | 0.760 | 0.905 | 0.650 | 2.53 |
| NLI | 0.370 | 0.610 | 0.750 | 0.525 | 8.64 |
| Physarum | 0.495 | 0.710 | 0.820 | 0.629 | 4.17 |
| CE+NLI | 0.540 | 0.820 | 0.955 | 0.696 | 2.13 |
| CE+NLI+Sim | 0.540 | 0.805 | 0.945 | 0.693 | 2.22 |
| CE+Physarum | 0.530 | 0.815 | 0.940 | 0.691 | 2.32 |
| NLI+Physarum | 0.550 | 0.765 | 0.885 | 0.687 | 2.54 |
| CE+NLI+Physarum | 0.590 | 0.845 | 0.965 | 0.734 | 1.96 |
| CE+NLI+Physarum+Sim | 0.590 | 0.845 | 0.965 | 0.734 | 1.96 |

Weights:
CE+NLI [5.381 1.877]
CE+NLI+Sim [3.596 1.878 3.161]
CE+Physarum [6.183 0.872]
NLI+Physarum [4.358 4.468]
CE+NLI+Physarum [5.001 1.899 1.018]
CE+NLI+Physarum+Sim [3.091 1.897 1.15  3.358]

--- Seed 199 ---


Test (d=50, seed=199): 100%|██████████| 200/200 [01:12<00:00,  2.77it/s]



| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.330 | 0.625 | 0.800 | 0.512 | 4.68 |
| CE | 0.460 | 0.765 | 0.905 | 0.630 | 2.64 |
| NLI | 0.350 | 0.610 | 0.765 | 0.510 | 8.70 |
| Physarum | 0.445 | 0.695 | 0.830 | 0.599 | 4.01 |
| CE+NLI | 0.500 | 0.800 | 0.935 | 0.665 | 2.29 |
| CE+NLI+Sim | 0.510 | 0.805 | 0.940 | 0.671 | 2.28 |
| CE+Physarum | 0.485 | 0.825 | 0.940 | 0.661 | 2.42 |
| NLI+Physarum | 0.500 | 0.800 | 0.895 | 0.665 | 2.60 |
| CE+NLI+Physarum | 0.535 | 0.870 | 0.960 | 0.705 | 2.03 |
| CE+NLI+Physarum+Sim | 0.540 | 0.870 | 0.955 | 0.707 | 2.03 |

Weights:
CE+NLI [5.315 1.818]
CE+NLI+Sim [3.765 1.826 2.703]
CE+Physarum [6.133 0.784]
NLI+Physarum [4.347 4.219]
CE+NLI+Physarum [4.94  1.861 1.   ]
CE+NLI+Physarum+Sim [3.258 1.859 1.118 2.93 ]

--- Seed 69 ---


Test (d=50, seed=69): 100%|██████████| 200/200 [01:12<00:00,  2.76it/s]


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.380 | 0.625 | 0.780 | 0.541 | 4.61 |
| CE | 0.485 | 0.780 | 0.890 | 0.648 | 2.62 |
| NLI | 0.390 | 0.615 | 0.755 | 0.535 | 8.66 |
| Physarum | 0.495 | 0.715 | 0.850 | 0.635 | 4.04 |
| CE+NLI | 0.535 | 0.810 | 0.935 | 0.688 | 2.21 |
| CE+NLI+Sim | 0.545 | 0.820 | 0.930 | 0.696 | 2.21 |
| CE+Physarum | 0.550 | 0.815 | 0.920 | 0.697 | 2.33 |
| NLI+Physarum | 0.560 | 0.760 | 0.890 | 0.690 | 2.54 |
| CE+NLI+Physarum | 0.595 | 0.845 | 0.950 | 0.732 | 2.00 |
| CE+NLI+Physarum+Sim | 0.600 | 0.850 | 0.950 | 0.736 | 1.98 |

Weights:
CE+NLI [5.365 1.815]
CE+NLI+Sim [3.762 1.823 2.844]
CE+Physarum [6.205 0.703]
NLI+Physarum [4.349 4.258]
CE+NLI+Physarum [5.01  1.856 0.956]
CE+NLI+Physarum+Sim [3.265 1.857 1.108 3.064]

=== AVERAGE RESULTS (d=50) ===

| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.355 | 0.623 | 0.792 | 0.528 | 4.62 |
| CE | 0.

In [ ]:
# dataset = load_dataset("lucadiliello/natural_questions_short_text", split="validation")

In [3]:
dataset = load_dataset("rajpurkar/squad_v2", split="validation[:1000]")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

{'id': '56ddde6b9a695914005b9628', 'title': 'Normans', 'context': 'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.', 'question': 'In what country is Normandy located?', 'answers': {'text': ['France', 'France', 'France', 'France'], 'answer_start': [159, 159, 159, 159]}}


In [4]:
dataset = load_dataset("rajpurkar/squad_v2", split="validation")

# filter out unanswerable questions
dataset = dataset.filter(lambda x: len(x['answers']['text']) > 0)

# take 1000
dataset = dataset.select(range(1000))

train_data = dataset.select(range(800))
test_data = dataset.select(range(800, 1000))

Filter:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [5]:
import numpy as np
import random
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from sklearn.linear_model import LogisticRegression

# =========================
# MODELS
# =========================
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
nli_pipeline = pipeline("text-classification", model="facebook/bart-large-mnli")
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# =========================
# DATASET
# =========================
dataset = load_dataset("rajpurkar/squad_v2", split="validation")
dataset = dataset.filter(lambda x: len(x['answers']['text']) > 0)
dataset = dataset.select(range(1000))

train_data = dataset.select(range(800))
test_data = dataset.select(range(800, 1000))

train_contexts = [d["context"] for d in train_data]
test_contexts = [d["context"] for d in test_data]

# =========================
# UTILS
# =========================
def normalize(x):
    x = np.array(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def get_rank(scores, labels):
    sorted_idx = np.argsort(scores)[::-1]
    ranks = np.where(labels[sorted_idx] == 1)[0]
    return ranks[0] + 1 if len(ranks) > 0 else len(scores)

def compute_metrics(ranks):
    ranks = np.array(ranks)
    return {
        "R@1": np.mean(ranks <= 1),
        "R@3": np.mean(ranks <= 3),
        "R@5": np.mean(ranks <= 5),
        "MRR": np.mean(1 / ranks),
        "Avg Rank": np.mean(ranks)
    }

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =========================
# CHUNKING
# =========================
def chunk_text(text, max_words=50, overlap=15):
    words = text.split()
    chunks = []
    step = max_words - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i+max_words])
        if len(chunk.strip()) > 0:
            chunks.append(chunk)
    return chunks

# =========================
# SCORING
# =========================
def get_chunk_embeddings(chunks):
    return bi_encoder.encode(chunks, convert_to_numpy=True)

def get_similarity_scores(q, chunks):
    q_emb = bi_encoder.encode([q], convert_to_numpy=True)[0]
    c_emb = get_chunk_embeddings(chunks)
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_emb = c_emb / (np.linalg.norm(c_emb, axis=1, keepdims=True) + 1e-8)
    return c_emb @ q_emb

def get_cross_scores(q, chunks):
    pairs = [[q, c] for c in chunks]
    return cross_encoder.predict(pairs)

# =========================
# NLI
# =========================
def get_nli_scores_batch(chunks, hypotheses, batch_size=16):
    inputs = [f"{c} </s> {h}" for c, h in zip(chunks, hypotheses)]
    outputs = nli_pipeline(
        inputs,
        batch_size=batch_size,
        truncation=True,
        top_k=None
    )
    scores = []
    for out in outputs:
        entail = 0.0
        if isinstance(out, list):
            for item in out:
                if isinstance(item, dict) and "ENTAIL" in item.get("label", "").upper():
                    entail = item.get("score", 0.0)
        elif isinstance(out, dict):
            if "ENTAIL" in out.get("label", "").upper():
                entail = out.get("score", 0.0)
        elif isinstance(out, str):
            if "ENTAIL" in out.upper():
                entail = 1.0
        scores.append(entail)
    return np.array(scores)

def get_evidence_scores_fast(query, chunks):
    answers = []
    confs = []
    for chunk in chunks:
        try:
            qa_out = qa_pipeline(question=query, context=chunk)
            answer = qa_out["answer"]
            conf = qa_out["score"]
            if conf < 0.3 or not answer.strip():
                answers.append("")
                confs.append(0.0)
            else:
                answers.append(answer)
                confs.append(conf)
        except:
            answers.append("")
            confs.append(0.0)

    hypotheses = [
        f"The answer is {a}." if a != "" else ""
        for a in answers
    ]

    valid_idx = [i for i, a in enumerate(answers) if a != ""]
    nli_scores = np.zeros(len(chunks))

    if len(valid_idx) > 0:
        valid_chunks = [chunks[i] for i in valid_idx]
        valid_hyp = [hypotheses[i] for i in valid_idx]
        entail_scores = get_nli_scores_batch(valid_chunks, valid_hyp)
        for idx, score in zip(valid_idx, entail_scores):
            nli_scores[idx] = score * confs[idx]

    return nli_scores

# =========================
# PHYSARUM v5
# =========================
def physarum_v5(sim_matrix, ce_scores, nli_scores,
                k=10, alpha=10.0, tau=0.5,
                lam_pos=0.2, lam_neg=0.15):
    N = len(ce_scores)

    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1
    knn_mask = np.maximum(knn_mask, knn_mask.T)
    sim_sparse = sim_matrix * knn_mask

    gate = 1 / (1 + np.exp(-alpha * (ce_scores - tau)))

    P_pos = np.zeros(N)
    P_neg = np.zeros(N)

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            if ce_scores[j] > ce_scores[i] + 0.05:
                P_pos[i] += sim_sparse[i, j] * ce_scores[j] * gate[i]
            if nli_scores[j] < 0.2:
                P_neg[i] += sim_sparse[i, j] * (0.5 - nli_scores[j]) * gate[i]

    return ce_scores + lam_pos * P_pos - lam_neg * P_neg

def get_physarum_scores_v5(q, chunks, sim, ce, nli):
    chunk_embs = get_chunk_embeddings(chunks)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-8)
    sim_matrix = chunk_embs @ chunk_embs.T
    sim_matrix = sim_matrix * sim[None, :]
    return physarum_v5(sim_matrix, ce, nli)

# =========================
# RAG DATASET
# =========================
def build_rag_dataset(data, num_distractors, distractor_pool):
    rag_data = []
    dropped = 0

    for sample in data:
        q = sample["question"]
        context = sample["context"]
        all_answers = sample["answers"]["text"]  # all annotator answers

        chunks = chunk_text(context)
        labels = [1 if any(ans in c for ans in all_answers) else 0 for c in chunks]

        if sum(labels) == 0:
            dropped += 1
            continue

        distractors = random.sample(distractor_pool, min(num_distractors, len(distractor_pool)))
        distractor_chunks = []
        for d in distractors:
            distractor_chunks.extend(chunk_text(d)[:1])

        chunks = chunks + distractor_chunks
        labels = labels + [0] * len(distractor_chunks)

        rag_data.append({
            "query": q,
            "chunks": chunks,
            "labels": labels
        })

    print(f"Dropped {dropped}/{len(data)} samples ({100*dropped/len(data):.1f}%)")
    return rag_data

# =========================
# CONFIG
# =========================
SEEDS = [42, 199, 69]
DISTRACTORS = [30, 50]

# =========================
# EXPERIMENT
# =========================
def run_single_experiment(train_data, test_data, train_contexts, test_contexts, num_distractors, seed):

    random.seed(seed)
    np.random.seed(seed)

    train_rag = build_rag_dataset(train_data, num_distractors, distractor_pool=train_contexts)
    test_rag = build_rag_dataset(test_data, num_distractors, distractor_pool=test_contexts)

    # ---------------------
    # TRAIN FEATURES
    # ---------------------
    feature_sets = {
        "CE+NLI": [],
        "CE+NLI+Sim": [],
        "CE+Physarum": [],
        "NLI+Physarum": [],
        "CE+NLI+Physarum": [],
        "CE+NLI+Physarum+Sim": []
    }
    y_train = []

    for sample in tqdm(train_rag, desc=f"Train (d={num_distractors}, seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        for i in range(len(chunks)):
            feature_sets["CE+NLI"].append([ce[i], nli[i]])
            feature_sets["CE+NLI+Sim"].append([ce[i], nli[i], sim[i]])
            feature_sets["CE+Physarum"].append([ce[i], phy[i]])
            feature_sets["NLI+Physarum"].append([nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum"].append([ce[i], nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum+Sim"].append([ce[i], nli[i], phy[i], sim[i]])
            y_train.append(labels[i])

    y_train = np.array(y_train)

    # ---------------------
    # TRAIN MODELS
    # ---------------------
    models = {}
    weights = {}
    for name, feats in feature_sets.items():
        X = np.array(feats)
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        model.fit(X, y_train)
        models[name] = model
        weights[name] = model.coef_[0]

    # ---------------------
    # EVALUATION
    # ---------------------
    results = {
        "Sim": [], "CE": [], "NLI": [], "Physarum": [],
        "CE+NLI": [], "CE+NLI+Sim": [],
        "CE+Physarum": [], "NLI+Physarum": [],
        "CE+NLI+Physarum": [], "CE+NLI+Physarum+Sim": []
    }

    for sample in tqdm(test_rag, desc=f"Test (d={num_distractors}, seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        results["Sim"].append(get_rank(sim, labels))
        results["CE"].append(get_rank(ce, labels))
        results["NLI"].append(get_rank(nli, labels))
        results["Physarum"].append(get_rank(phy, labels))

        feat_map = {
            "CE+NLI": np.stack([ce, nli], axis=1),
            "CE+NLI+Sim": np.stack([ce, nli, sim], axis=1),
            "CE+Physarum": np.stack([ce, phy], axis=1),
            "NLI+Physarum": np.stack([nli, phy], axis=1),
            "CE+NLI+Physarum": np.stack([ce, nli, phy], axis=1),
            "CE+NLI+Physarum+Sim": np.stack([ce, nli, phy, sim], axis=1),
        }

        for name, feat in feat_map.items():
            scores = models[name].predict_proba(feat)[:, 1]
            results[name].append(get_rank(scores, labels))

    metrics = {k: compute_metrics(v) for k, v in results.items()}
    return metrics, weights


# =========================
# MAIN LOOP
# =========================
all_results = {}

for num_distractors in DISTRACTORS:
    print(f"\n{'='*20} DISTRACTORS = {num_distractors} {'='*20}")
    seed_metrics = []

    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")
        metrics, weights = run_single_experiment(
            train_data, test_data,
            train_contexts, test_contexts,
            num_distractors, seed
        )
        seed_metrics.append(metrics)

        # per-seed table
        print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
        print(f"|--------|-----|-----|-----|-----|----------|")
        for method, m in metrics.items():
            print(f"| {method} | {m['R@1']:.3f} | {m['R@3']:.3f} | {m['R@5']:.3f} | {m['MRR']:.3f} | {m['Avg Rank']:.2f} |")

        print("\nWeights:")
        for name, w in weights.items():
            print(f"  {name} {np.round(w, 3)}")

    # average across seeds
    print(f"\n=== AVERAGE RESULTS (d={num_distractors}) ===")
    print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
    print(f"|--------|-----|-----|-----|-----|----------|")
    methods = list(seed_metrics[0].keys())
    for method in methods:
        avg = {k: np.mean([s[method][k] for s in seed_metrics]) for k in ["R@1", "R@3", "R@5", "MRR", "Avg Rank"]}
        print(f"| {method} | {avg['R@1']:.3f} | {avg['R@3']:.3f} | {avg['R@5']:.3f} | {avg['MRR']:.3f} | {avg['Avg Rank']:.2f} |")

    all_results[num_distractors] = seed_metrics

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.470 | 0.830 | 0.920 | 0.659 | 2.69 |
| CE | 0.560 | 0.880 | 0.950 | 0.721 | 2.48 |
| NLI | 0.490 | 0.795 | 0.860 | 0.645 | 8.46 |
| Physarum | 0.455 | 0.705 | 0.790 | 0.598 | 4.95 |
| CE+NLI | 0.580 | 0.895 | 0.960 | 0.738 | 2.19 |
| CE+NLI+Sim | 0.595 | 0.890 | 0.955 | 0.747 | 2.11 |
| CE+Physarum | 0.575 | 0.885 | 0.935 | 0.730 | 2.43 |
| NLI+Physarum | 0.550 | 0.785 | 0.850 | 0.680 | 3.80 |
| CE+NLI+Physarum | 0.580 | 0.895 | 0.960 | 0.737 | 2.12 |
| CE+NLI+Physarum+Sim | 0.620 | 0.895 | 0.960 | 0.763 | 2.02 |

Weights:
  CE+NLI [6.916 1.8  ]
  CE+NLI+Sim [1.978 2.022 7.614]
  CE+Physarum [5.644 2.256]
  NLI+Physarum [2.204 8.097]
  CE+NLI+Physarum [4.145 1.826 3.276]
  CE+NLI+Physarum+Sim [1.398 1.993 0.682 7.594]

--- Seed 199 ---
Dropped 0/800 samples (0.0%)
Dropped 0/200 samples (0.0%)


Test (d=50, seed=199): 100%|██████████| 200/200 [01:15<00:00,  2.64it/s]



| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.485 | 0.830 | 0.920 | 0.664 | 2.65 |
| CE | 0.565 | 0.885 | 0.935 | 0.725 | 2.44 |
| NLI | 0.510 | 0.795 | 0.860 | 0.653 | 8.45 |
| Physarum | 0.485 | 0.715 | 0.805 | 0.623 | 5.08 |
| CE+NLI | 0.605 | 0.895 | 0.955 | 0.755 | 2.15 |
| CE+NLI+Sim | 0.605 | 0.885 | 0.945 | 0.751 | 2.08 |
| CE+Physarum | 0.610 | 0.885 | 0.930 | 0.752 | 2.48 |
| NLI+Physarum | 0.585 | 0.810 | 0.850 | 0.709 | 3.77 |
| CE+NLI+Physarum | 0.645 | 0.915 | 0.950 | 0.783 | 2.00 |
| CE+NLI+Physarum+Sim | 0.645 | 0.895 | 0.950 | 0.779 | 1.94 |

Weights:
  CE+NLI [6.92  1.769]
  CE+NLI+Sim [2.004 1.95  7.572]
  CE+Physarum [5.601 2.292]
  NLI+Physarum [2.205 8.097]
  CE+NLI+Physarum [4.143 1.811 3.282]
  CE+NLI+Physarum+Sim [1.392 1.966 0.723 7.544]

--- Seed 69 ---
Dropped 0/800 samples (0.0%)
Dropped 0/200 samples (0.0%)


Test (d=50, seed=69): 100%|██████████| 200/200 [01:16<00:00,  2.61it/s]


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.505 | 0.855 | 0.925 | 0.682 | 2.60 |
| CE | 0.595 | 0.895 | 0.950 | 0.746 | 2.35 |
| NLI | 0.530 | 0.810 | 0.860 | 0.673 | 8.37 |
| Physarum | 0.480 | 0.735 | 0.805 | 0.620 | 5.08 |
| CE+NLI | 0.630 | 0.935 | 0.965 | 0.776 | 2.02 |
| CE+NLI+Sim | 0.640 | 0.915 | 0.965 | 0.780 | 1.93 |
| CE+Physarum | 0.645 | 0.900 | 0.935 | 0.775 | 2.31 |
| NLI+Physarum | 0.580 | 0.810 | 0.875 | 0.708 | 3.96 |
| CE+NLI+Physarum | 0.635 | 0.925 | 0.960 | 0.777 | 1.99 |
| CE+NLI+Physarum+Sim | 0.670 | 0.925 | 0.970 | 0.797 | 1.82 |

Weights:
  CE+NLI [6.925 1.958]
  CE+NLI+Sim [1.975 2.12  7.62 ]
  CE+Physarum [5.857 2.034]
  NLI+Physarum [2.389 8.13 ]
  CE+NLI+Physarum [4.61  2.014 2.705]
  CE+NLI+Physarum+Sim [1.446 2.112 0.65  7.618]

=== AVERAGE RESULTS (d=50) ===

| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.487 | 0.838 | 0.922 | 0.668 | 2.65

200

In [5]:
import numpy as np
import random
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from sklearn.linear_model import LogisticRegression

# =========================
# MODELS
# =========================

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
nli_pipeline = pipeline("text-classification", model="facebook/bart-large-mnli")
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# =========================
# DATASET
# =========================
dataset = load_dataset("rajpurkar/squad_v2", split="validation")
dataset = dataset.filter(lambda x: len(x['answers']['text']) > 0)
dataset = dataset.select(range(1000))

train_data = dataset.select(range(800))
test_data = dataset.select(range(800, 1000))

train_contexts = [d["context"] for d in train_data]
test_contexts = [d["context"] for d in test_data]

# =========================
# UTILS
# =========================
def normalize(x):
    x = np.array(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def get_rank(scores, labels):
    sorted_idx = np.argsort(scores)[::-1]
    ranks = np.where(labels[sorted_idx] == 1)[0]
    return ranks[0] + 1 if len(ranks) > 0 else len(scores)

def compute_metrics(ranks):
    ranks = np.array(ranks)
    return {
        "R@1": np.mean(ranks <= 1),
        "R@3": np.mean(ranks <= 3),
        "R@5": np.mean(ranks <= 5),
        "MRR": np.mean(1 / ranks),
        "Avg Rank": np.mean(ranks)
    }

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =========================
# CHUNKING
# =========================
def chunk_text(text, max_words=50, overlap=15):
    words = text.split()
    chunks = []
    step = max_words - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i+max_words])
        if len(chunk.strip()) > 0:
            chunks.append(chunk)
    return chunks

# def get_hard_negatives(query, pool, k=50, sample_size=500):
#     # sample a subset (for speed)
#     subset = random.sample(pool, min(sample_size, len(pool)))

#     # encode once (batch = fast)
#     q_emb = bi_encoder.encode([query], convert_to_numpy=True)[0]
#     ctx_embs = bi_encoder.encode(subset, convert_to_numpy=True)

#     # normalize
#     q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
#     ctx_embs = ctx_embs / (np.linalg.norm(ctx_embs, axis=1, keepdims=True) + 1e-8)

#     # cosine similarity
#     scores = ctx_embs @ q_emb

#     # sort descending
#     top_idx = np.argsort(scores)[::-1][:k]

#     return [subset[i] for i in top_idx]

# pool_embs = bi_encoder.encode(distractor_pool, convert_to_numpy=True)
# pool_embs = pool_embs / (np.linalg.norm(pool_embs, axis=1, keepdims=True) + 1e-8)

def get_hard_negatives_fast(query, pool, pool_embs, k=50, threshold=0.3):
    q_emb = bi_encoder.encode([query], convert_to_numpy=True)[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)

    scores = pool_embs @ q_emb
    top_idx = np.argsort(scores)[::-1]

    # ---- FILTER HARD NEGATIVES ----
    filtered_idx = [i for i in top_idx if scores[i] > threshold][:k]

    # ---- FALLBACK (important) ----
    if len(filtered_idx) < k:
        filtered_idx = top_idx[:k]

    return [pool[i] for i in filtered_idx]

# =========================
# SCORING
# =========================
def get_chunk_embeddings(chunks):
    return bi_encoder.encode(chunks, convert_to_numpy=True)

def get_similarity_scores(q, chunks):
    q_emb = bi_encoder.encode([q], convert_to_numpy=True)[0]
    c_emb = get_chunk_embeddings(chunks)
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_emb = c_emb / (np.linalg.norm(c_emb, axis=1, keepdims=True) + 1e-8)
    return c_emb @ q_emb

def get_cross_scores(q, chunks):
    pairs = [[q, c] for c in chunks]
    return cross_encoder.predict(pairs)

# =========================
# NLI
# =========================
def get_nli_scores_batch(chunks, hypotheses, batch_size=16):
    inputs = [f"{c} </s> {h}" for c, h in zip(chunks, hypotheses)]
    outputs = nli_pipeline(
        inputs,
        batch_size=batch_size,
        truncation=True,
        top_k=None
    )
    scores = []
    for out in outputs:
        entail = 0.0
        if isinstance(out, list):
            for item in out:
                if isinstance(item, dict) and "ENTAIL" in item.get("label", "").upper():
                    entail = item.get("score", 0.0)
        elif isinstance(out, dict):
            if "ENTAIL" in out.get("label", "").upper():
                entail = out.get("score", 0.0)
        elif isinstance(out, str):
            if "ENTAIL" in out.upper():
                entail = 1.0
        scores.append(entail)
    return np.array(scores)

def get_evidence_scores_fast(query, chunks):
    answers = []
    confs = []
    for chunk in chunks:
        try:
            qa_out = qa_pipeline(question=query, context=chunk)
            answer = qa_out["answer"]
            conf = qa_out["score"]
            if conf < 0.3 or not answer.strip():
                answers.append("")
                confs.append(0.0)
            else:
                answers.append(answer)
                confs.append(conf)
        except:
            answers.append("")
            confs.append(0.0)

    hypotheses = [
        f"The answer is {a}." if a != "" else ""
        for a in answers
    ]

    valid_idx = [i for i, a in enumerate(answers) if a != ""]
    nli_scores = np.zeros(len(chunks))

    if len(valid_idx) > 0:
        valid_chunks = [chunks[i] for i in valid_idx]
        valid_hyp = [hypotheses[i] for i in valid_idx]
        entail_scores = get_nli_scores_batch(valid_chunks, valid_hyp)
        for idx, score in zip(valid_idx, entail_scores):
            nli_scores[idx] = score * confs[idx]

    return nli_scores

# =========================
# PHYSARUM v5
# =========================
def physarum_v5(sim_matrix, ce_scores, nli_scores,
                k=10, alpha=10.0, tau=0.5,
                lam_pos=0.2, lam_neg=0.15):
    N = len(ce_scores)

    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1
    knn_mask = np.maximum(knn_mask, knn_mask.T)
    sim_sparse = sim_matrix * knn_mask

    gate = 1 / (1 + np.exp(-alpha * (ce_scores - tau)))

    P_pos = np.zeros(N)
    P_neg = np.zeros(N)

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            if ce_scores[j] > ce_scores[i] + 0.05:
                P_pos[i] += sim_sparse[i, j] * ce_scores[j] * gate[i]
            if nli_scores[j] < 0.2:
                P_neg[i] += sim_sparse[i, j] * (0.5 - nli_scores[j]) * gate[i]

    return ce_scores + lam_pos * P_pos - lam_neg * P_neg

def get_physarum_scores_v5(q, chunks, sim, ce, nli):
    chunk_embs = get_chunk_embeddings(chunks)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-8)
    sim_matrix = chunk_embs @ chunk_embs.T
    sim_matrix = sim_matrix * sim[None, :]
    return physarum_v5(sim_matrix, ce, nli)


def physarum_v6(sim_matrix, ce_scores, nli_scores,
                k=10,
                alpha=8.0,        # gating sharpness
                tau=0.5,          # CE confidence threshold
                lam=0.15,         # overall graph strength (SMALL)
                beta=0.5,         # NLI influence
                iters=2):         # number of propagation steps

    N = len(ce_scores)

    # ---- Step 1: build kNN graph (sparse) ----
    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1

    knn_mask = np.maximum(knn_mask, knn_mask.T)
    W = sim_matrix * knn_mask

    # ---- Step 2: normalize rows (VERY IMPORTANT) ----
    row_sum = W.sum(axis=1, keepdims=True) + 1e-8
    W = W / row_sum

    # ---- Step 3: initialize scores ----
    s = ce_scores.copy()

    for _ in range(iters):

        new_s = np.zeros(N)

        for i in range(N):

            # ---- gating: trust only strong nodes ----
            gate_i = 1 / (1 + np.exp(-alpha * (s[i] - tau)))

            msg = 0.0

            for j in range(N):
                if i == j or W[i, j] == 0:
                    continue

                # ---- only propagate from confident nodes ----
                if s[j] < tau:
                    continue

                # ---- edge weight: similarity × confidence ----
                weight = W[i, j] * s[j]

                # ---- NLI modulation ----
                # positive if entailment high, negative if low
                nli_effect = (nli_scores[j] - 0.5)

                msg += weight * (1 + beta * nli_effect)

            # ---- combine with original CE (RESIDUAL CONNECTION) ----
            new_s[i] = (1 - lam) * ce_scores[i] + lam * gate_i * msg

        s = new_s

    return s

def get_physarum_scores_v6(q, chunks, sim, ce, nli):
    chunk_embs = get_chunk_embeddings(chunks)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-8)

    sim_matrix = chunk_embs @ chunk_embs.T

    # inject query relevance
    sim_matrix = sim_matrix * sim[None, :]

    return physarum_v6(sim_matrix, ce, nli)

# =========================
# RAG DATASET
# =========================
def build_rag_dataset(data, num_distractors, distractor_pool, pool_embs):
    rag_data = []
    dropped = 0

    for sample in data:
        q = sample["question"]
        context = sample["context"]
        all_answers = sample["answers"]["text"]

        # ---- positive chunks ----
        chunks = chunk_text(context)
        labels = [1 if any(ans in c for ans in all_answers) else 0 for c in chunks]

        if sum(labels) == 0:
            dropped += 1
            continue

        # ---- HARD NEGATIVES ----
        hard_negatives = get_hard_negatives_fast(
            q, distractor_pool, pool_embs, k=num_distractors
        )

        distractor_chunks = []
        for d in hard_negatives:
            distractor_chunks.extend(chunk_text(d)[:1])

        # ---- combine ----
        chunks = chunks + distractor_chunks
        labels = labels + [0] * len(distractor_chunks)

        # ---- store ----
        rag_data.append({
            "query": q,
            "chunks": chunks,
            "labels": labels
        })

    print(f"Dropped {dropped}/{len(data)} samples ({100*dropped/len(data):.1f}%)")
    return rag_data

# =========================
# CONFIG
# =========================
SEEDS = [42, 199, 69]
DISTRACTORS = [30, 50]

# =========================
# EXPERIMENT
# =========================
def run_single_experiment(train_data, test_data, train_contexts, test_contexts, num_distractors, seed):

    random.seed(seed)
    np.random.seed(seed)
    train_pool_embs = bi_encoder.encode(train_contexts, convert_to_numpy=True)
    train_pool_embs = train_pool_embs / (np.linalg.norm(train_pool_embs, axis=1, keepdims=True) + 1e-8)
    
    test_pool_embs = bi_encoder.encode(test_contexts, convert_to_numpy=True)
    test_pool_embs = test_pool_embs / (np.linalg.norm(test_pool_embs, axis=1, keepdims=True) + 1e-8)
    train_rag = build_rag_dataset(
        train_data,
        num_distractors,
        distractor_pool=train_contexts,
        pool_embs=train_pool_embs
    )
    
    test_rag = build_rag_dataset(
        test_data,
        num_distractors,
        distractor_pool=test_contexts,
        pool_embs=test_pool_embs
    )

    # ---------------------
    # TRAIN FEATURES
    # ---------------------
    feature_sets = {
        "CE+NLI": [],
        "CE+NLI+Sim": [],
        "CE+Physarum": [],
        "NLI+Physarum": [],
        "CE+NLI+Physarum": [],
        "CE+NLI+Physarum+Sim": []
    }
    y_train = []

    for sample in tqdm(train_rag, desc=f"Train (d={num_distractors}, seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v6(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        for i in range(len(chunks)):
            feature_sets["CE+NLI"].append([ce[i], nli[i]])
            feature_sets["CE+NLI+Sim"].append([ce[i], nli[i], sim[i]])
            feature_sets["CE+Physarum"].append([ce[i], phy[i]])
            feature_sets["NLI+Physarum"].append([nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum"].append([ce[i], nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum+Sim"].append([ce[i], nli[i], phy[i], sim[i]])
            y_train.append(labels[i])

    y_train = np.array(y_train)

    # ---------------------
    # TRAIN MODELS
    # ---------------------
    models = {}
    weights = {}
    for name, feats in feature_sets.items():
        X = np.array(feats)
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        model.fit(X, y_train)
        models[name] = model
        weights[name] = model.coef_[0]

    # ---------------------
    # EVALUATION
    # ---------------------
    results = {
        "Sim": [], "CE": [], "NLI": [], "Physarum": [],
        "CE+NLI": [], "CE+NLI+Sim": [],
        "CE+Physarum": [], "NLI+Physarum": [],
        "CE+NLI+Physarum": [], "CE+NLI+Physarum+Sim": []
    }

    for sample in tqdm(test_rag, desc=f"Test (d={num_distractors}, seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = ce > 0.2
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v6(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        results["Sim"].append(get_rank(sim, labels))
        results["CE"].append(get_rank(ce, labels))
        results["NLI"].append(get_rank(nli, labels))
        results["Physarum"].append(get_rank(phy, labels))

        feat_map = {
            "CE+NLI": np.stack([ce, nli], axis=1),
            "CE+NLI+Sim": np.stack([ce, nli, sim], axis=1),
            "CE+Physarum": np.stack([ce, phy], axis=1),
            "NLI+Physarum": np.stack([nli, phy], axis=1),
            "CE+NLI+Physarum": np.stack([ce, nli, phy], axis=1),
            "CE+NLI+Physarum+Sim": np.stack([ce, nli, phy, sim], axis=1),
        }

        for name, feat in feat_map.items():
            scores = models[name].predict_proba(feat)[:, 1]
            results[name].append(get_rank(scores, labels))

    metrics = {k: compute_metrics(v) for k, v in results.items()}
    return metrics, weights


# =========================
# MAIN LOOP
# =========================
all_results = {}

for num_distractors in DISTRACTORS:
    print(f"\n{'='*20} DISTRACTORS = {num_distractors} {'='*20}")
    seed_metrics = []

    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")
        metrics, weights = run_single_experiment(
            train_data, test_data,
            train_contexts, test_contexts,
            num_distractors, seed
        )
        seed_metrics.append(metrics)

        # per-seed table
        print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
        print(f"|--------|-----|-----|-----|-----|----------|")
        for method, m in metrics.items():
            print(f"| {method} | {m['R@1']:.3f} | {m['R@3']:.3f} | {m['R@5']:.3f} | {m['MRR']:.3f} | {m['Avg Rank']:.2f} |")

        print("\nWeights:")
        for name, w in weights.items():
            print(f"  {name} {np.round(w, 3)}")

    # average across seeds
    print(f"\n=== AVERAGE RESULTS (d={num_distractors}) ===")
    print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
    print(f"|--------|-----|-----|-----|-----|----------|")
    methods = list(seed_metrics[0].keys())
    for method in methods:
        avg = {k: np.mean([s[method][k] for s in seed_metrics]) for k in ["R@1", "R@3", "R@5", "MRR", "Avg Rank"]}
        print(f"| {method} | {avg['R@1']:.3f} | {avg['R@3']:.3f} | {avg['R@5']:.3f} | {avg['MRR']:.3f} | {avg['Avg Rank']:.2f} |")

    all_results[num_distractors] = seed_metrics

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.375 | 0.480 | 0.655 | 0.507 | 5.82 |
| CE | 0.420 | 0.505 | 0.725 | 0.549 | 4.75 |
| NLI | 0.395 | 0.485 | 0.670 | 0.513 | 9.74 |
| Physarum | 0.505 | 0.630 | 0.750 | 0.622 | 4.37 |
| CE+NLI | 0.420 | 0.520 | 0.745 | 0.556 | 4.17 |
| CE+NLI+Sim | 0.455 | 0.550 | 0.765 | 0.583 | 4.02 |
| CE+Physarum | 0.465 | 0.580 | 0.760 | 0.587 | 4.58 |
| NLI+Physarum | 0.520 | 0.655 | 0.765 | 0.638 | 3.83 |
| CE+NLI+Physarum | 0.495 | 0.610 | 0.765 | 0.614 | 3.83 |
| CE+NLI+Physarum+Sim | 0.485 | 0.605 | 0.770 | 0.607 | 3.90 |

Weights:
  CE+NLI [3.953 1.291]
  CE+NLI+Sim [3.527 1.274 0.614]
  CE+Physarum [12.663 -8.765]
  NLI+Physarum [1.347 4.259]
  CE+NLI+Physarum [11.451  1.276 -8.291]
  CE+NLI+Physarum+Sim [10.672  1.259 -7.756  0.448]

--- Seed 199 ---
Dropped 0/800 samples (0.0%)
Dropped 0/200 samples (0.0%)


Test (d=50, seed=199): 100%|██████████| 200/200 [01:51<00:00,  1.79it/s]



| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.375 | 0.480 | 0.655 | 0.507 | 5.82 |
| CE | 0.420 | 0.505 | 0.725 | 0.549 | 4.75 |
| NLI | 0.395 | 0.485 | 0.670 | 0.513 | 9.74 |
| Physarum | 0.505 | 0.630 | 0.750 | 0.622 | 4.37 |
| CE+NLI | 0.420 | 0.520 | 0.745 | 0.556 | 4.17 |
| CE+NLI+Sim | 0.455 | 0.550 | 0.765 | 0.583 | 4.02 |
| CE+Physarum | 0.465 | 0.580 | 0.760 | 0.587 | 4.58 |
| NLI+Physarum | 0.520 | 0.655 | 0.765 | 0.638 | 3.83 |
| CE+NLI+Physarum | 0.495 | 0.610 | 0.765 | 0.614 | 3.83 |
| CE+NLI+Physarum+Sim | 0.485 | 0.605 | 0.770 | 0.607 | 3.90 |

Weights:
  CE+NLI [3.953 1.291]
  CE+NLI+Sim [3.527 1.274 0.614]
  CE+Physarum [12.663 -8.765]
  NLI+Physarum [1.347 4.259]
  CE+NLI+Physarum [11.451  1.276 -8.291]
  CE+NLI+Physarum+Sim [10.672  1.259 -7.756  0.448]

--- Seed 69 ---
Dropped 0/800 samples (0.0%)
Dropped 0/200 samples (0.0%)


Test (d=50, seed=69): 100%|██████████| 200/200 [01:50<00:00,  1.81it/s]


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.375 | 0.480 | 0.655 | 0.507 | 5.82 |
| CE | 0.420 | 0.505 | 0.725 | 0.549 | 4.75 |
| NLI | 0.395 | 0.485 | 0.670 | 0.513 | 9.74 |
| Physarum | 0.505 | 0.630 | 0.750 | 0.622 | 4.37 |
| CE+NLI | 0.420 | 0.520 | 0.745 | 0.556 | 4.17 |
| CE+NLI+Sim | 0.455 | 0.550 | 0.765 | 0.583 | 4.02 |
| CE+Physarum | 0.465 | 0.580 | 0.760 | 0.587 | 4.58 |
| NLI+Physarum | 0.520 | 0.655 | 0.765 | 0.638 | 3.83 |
| CE+NLI+Physarum | 0.495 | 0.610 | 0.765 | 0.614 | 3.83 |
| CE+NLI+Physarum+Sim | 0.485 | 0.605 | 0.770 | 0.607 | 3.90 |

Weights:
  CE+NLI [3.953 1.291]
  CE+NLI+Sim [3.527 1.274 0.614]
  CE+Physarum [12.663 -8.765]
  NLI+Physarum [1.347 4.259]
  CE+NLI+Physarum [11.451  1.276 -8.291]
  CE+NLI+Physarum+Sim [10.672  1.259 -7.756  0.448]

=== AVERAGE RESULTS (d=50) ===

| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.375 | 0.480 | 0.655 | 0.5

In [1]:
import numpy as np
import random
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from sklearn.linear_model import LogisticRegression

# =========================
# MODELS
# =========================
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
nli_pipeline = pipeline("text-classification", model="facebook/bart-large-mnli")
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# =========================
# DATASET (SQuAD v2)
# =========================
dataset = load_dataset("squad_v2", split="train[:1000]")

# remove unanswerable for now (simpler)
dataset = dataset.filter(lambda x: len(x["answers"]["text"]) > 0)

n = len(dataset)
train_size = int(0.8 * n)

train_data = dataset.select(range(train_size))
test_data = dataset.select(range(train_size, n))



# =========================
# UTILS
# =========================
def normalize(x):
    x = np.array(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def get_rank(scores, labels):
    sorted_idx = np.argsort(scores)[::-1]
    ranks = np.where(labels[sorted_idx] == 1)[0]
    return ranks[0] + 1 if len(ranks) > 0 else len(scores)

def compute_metrics(ranks):
    ranks = np.array(ranks)
    return {
        "R@1": np.mean(ranks <= 1),
        "R@3": np.mean(ranks <= 3),
        "R@5": np.mean(ranks <= 5),
        "MRR": np.mean(1 / ranks),
        "Avg Rank": np.mean(ranks)
    }

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =========================
# SCORING
# =========================
def get_chunk_embeddings(chunks):
    return bi_encoder.encode(chunks, convert_to_numpy=True)

def get_similarity_scores(q, chunks):
    q_emb = bi_encoder.encode([q], convert_to_numpy=True)[0]
    c_emb = get_chunk_embeddings(chunks)
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_emb = c_emb / (np.linalg.norm(c_emb, axis=1, keepdims=True) + 1e-8)
    return c_emb @ q_emb

def get_cross_scores(q, chunks):
    pairs = [[q, c] for c in chunks]
    return cross_encoder.predict(pairs)

# =========================
# NLI
# =========================
def get_nli_scores_batch(chunks, hypotheses, batch_size=16):
    inputs = [f"{c} </s> {h}" for c, h in zip(chunks, hypotheses)]
    outputs = nli_pipeline(
        inputs,
        batch_size=batch_size,
        truncation=True,
        top_k=None
    )
    scores = []
    for out in outputs:
        entail = 0.0
        if isinstance(out, list):
            for item in out:
                if isinstance(item, dict) and "ENTAIL" in item.get("label", "").upper():
                    entail = item.get("score", 0.0)
        elif isinstance(out, dict):
            if "ENTAIL" in out.get("label", "").upper():
                entail = out.get("score", 0.0)
        elif isinstance(out, str):
            if "ENTAIL" in out.upper():
                entail = 1.0
        scores.append(entail)
    return np.array(scores)

def get_evidence_scores_fast(query, chunks):
    answers = []
    confs = []
    for chunk in chunks:
        try:
            qa_out = qa_pipeline(question=query, context=chunk)
            answer = qa_out["answer"]
            conf = qa_out["score"]
            if conf < 0.3 or not answer.strip():
                answers.append("")
                confs.append(0.0)
            else:
                answers.append(answer)
                confs.append(conf)
        except:
            answers.append("")
            confs.append(0.0)

    hypotheses = [
        f"The answer is {a}." if a != "" else ""
        for a in answers
    ]

    valid_idx = [i for i, a in enumerate(answers) if a != ""]
    nli_scores = np.zeros(len(chunks))

    if len(valid_idx) > 0:
        valid_chunks = [chunks[i] for i in valid_idx]
        valid_hyp = [hypotheses[i] for i in valid_idx]
        entail_scores = get_nli_scores_batch(valid_chunks, valid_hyp)
        for idx, score in zip(valid_idx, entail_scores):
            nli_scores[idx] = score * confs[idx]

    return nli_scores

# =========================
# PHYSARUM v5
# =========================

def physarum_diversity(sim_matrix, ce_scores, k=5, lam=0.3):
    N = len(ce_scores)
    k = min(k, N-1)
    
    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1
    knn_mask = np.maximum(knn_mask, knn_mask.T)
    sim_sparse = sim_matrix * knn_mask

    penalty = np.zeros(N)
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            # if j is a stronger neighbour, i gets penalized
            if ce_scores[j] > ce_scores[i] and sim_sparse[i, j] > 0:
                penalty[i] += sim_sparse[i, j] * (ce_scores[j] - ce_scores[i])

    return ce_scores - lam * penalty

def physarum_v5(sim_matrix, ce_scores,
                k=5, alpha=10.0, tau=0.5,
                lam_pos=0.3, lam_neg=0.2):
    N = len(ce_scores)

    # kNN with k < N so sparsity actually does something
    k = min(k, N - 1)
    knn_mask = np.zeros_like(sim_matrix)
    for i in range(N):
        top_k_idx = np.argsort(sim_matrix[i])[-k:]
        knn_mask[i, top_k_idx] = 1
    knn_mask = np.maximum(knn_mask, knn_mask.T)
    sim_sparse = sim_matrix * knn_mask

    gate = 1 / (1 + np.exp(-alpha * (ce_scores - tau)))

    P_pos = np.zeros(N)
    P_neg = np.zeros(N)

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            diff = ce_scores[j] - ce_scores[i]
            # positive: pull up nodes near high-CE neighbours
            if diff > 0.05:
                P_pos[i] += sim_sparse[i, j] * diff * gate[j]
            # negative: push down nodes near low-CE neighbours
            if diff < -0.05:
                P_neg[i] += sim_sparse[i, j] * abs(diff) * (1 - gate[j])

    return ce_scores + lam_pos * P_pos - lam_neg * P_neg
    

def get_physarum_scores_v5(q, chunks, sim, ce, nli):


    chunk_embs = get_chunk_embeddings(chunks)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-8)
    sim_matrix = chunk_embs @ chunk_embs.T
    # symmetric query injection this time
    sim_matrix = sim_matrix * sim[:, None] * sim[None, :]
    return physarum_diversity(sim_matrix, ce)
# =========================
# BUILD RAG DATASET
# =========================
# =========================
# BUILD RAG DATASET (SQuAD)
# =========================
def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def build_rag_dataset(data):
    rag_data = []

    for sample in data:
        q = sample["question"]
        context = sample["context"]
        answers = sample["answers"]["text"]

        if len(answers) == 0:
            continue

        chunks = chunk_text(context)

        labels = []
        for chunk in chunks:
            if any(ans.lower() in chunk.lower() for ans in answers):
                labels.append(1)
            else:
                labels.append(0)

        # keep same assumption as MS MARCO (1 positive)
        if sum(labels) != 1:
            continue

        rag_data.append({
            "query": q,
            "chunks": chunks,
            "labels": labels
        })

    print(f"Built {len(rag_data)}/{len(data)} samples")
    return rag_data

# =========================
# CONFIG
# =========================
SEEDS = [42]

# =========================
# EXPERIMENT
# =========================
def run_single_experiment(train_data, test_data, seed):

    random.seed(seed)
    np.random.seed(seed)

    train_rag = build_rag_dataset(train_data)
    test_rag = build_rag_dataset(test_data)

    # ---------------------
    # TRAIN FEATURES
    # ---------------------
    feature_sets = {
        "CE+NLI": [],
        "CE+NLI+Sim": [],
        "CE+Physarum": [],
        "NLI+Physarum": [],
        "CE+NLI+Physarum": [],
        "CE+NLI+Physarum+Sim": []
    }
    y_train = []

    for sample in tqdm(train_rag, desc=f"Train (seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = np.ones(len(chunks), dtype=bool)
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        for i in range(len(chunks)):
            feature_sets["CE+NLI"].append([ce[i], nli[i]])
            feature_sets["CE+NLI+Sim"].append([ce[i], nli[i], sim[i]])
            feature_sets["CE+Physarum"].append([ce[i], phy[i]])
            feature_sets["NLI+Physarum"].append([nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum"].append([ce[i], nli[i], phy[i]])
            feature_sets["CE+NLI+Physarum+Sim"].append([ce[i], nli[i], phy[i], sim[i]])
            y_train.append(labels[i])

    y_train = np.array(y_train)

    # ---------------------
    # TRAIN MODELS
    # ---------------------
    models = {}
    weights = {}
    for name, feats in feature_sets.items():
        X = np.array(feats)
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        model.fit(X, y_train)
        models[name] = model
        weights[name] = model.coef_[0]

    # ---------------------
    # EVALUATION
    # ---------------------
    results = {
        "Sim": [], "CE": [], "NLI": [], "Physarum": [],
        "CE+NLI": [], "CE+NLI+Sim": [],
        "CE+Physarum": [], "NLI+Physarum": [],
        "CE+NLI+Physarum": [], "CE+NLI+Physarum+Sim": []
    }

    for sample in tqdm(test_rag, desc=f"Test (seed={seed})"):
        q = sample["query"]
        chunks = sample["chunks"]
        labels = np.array(sample["labels"])

        sim = normalize(get_similarity_scores(q, chunks))
        ce = normalize(get_cross_scores(q, chunks))

        mask = np.ones(len(chunks), dtype=bool)
        nli_raw = np.zeros(len(chunks))
        if mask.sum() > 0:
            selected_chunks = [chunks[i] for i in range(len(chunks)) if mask[i]]
            selected_scores = get_evidence_scores_fast(q, selected_chunks)
            idx = 0
            for i in range(len(chunks)):
                if mask[i]:
                    nli_raw[i] = selected_scores[idx]
                    idx += 1

        nli = normalize(nli_raw)
        phy = get_physarum_scores_v5(q, chunks, sim=sim, ce=ce, nli=nli_raw)

        results["Sim"].append(get_rank(sim, labels))
        results["CE"].append(get_rank(ce, labels))
        results["NLI"].append(get_rank(nli, labels))
        results["Physarum"].append(get_rank(phy, labels))

        feat_map = {
            "CE+NLI": np.stack([ce, nli], axis=1),
            "CE+NLI+Sim": np.stack([ce, nli, sim], axis=1),
            "CE+Physarum": np.stack([ce, phy], axis=1),
            "NLI+Physarum": np.stack([nli, phy], axis=1),
            "CE+NLI+Physarum": np.stack([ce, nli, phy], axis=1),
            "CE+NLI+Physarum+Sim": np.stack([ce, nli, phy, sim], axis=1),
        }

        for name, feat in feat_map.items():
            scores = models[name].predict_proba(feat)[:, 1]
            results[name].append(get_rank(scores, labels))

    metrics = {k: compute_metrics(v) for k, v in results.items()}
    return metrics, weights


# =========================
# MAIN LOOP
# =========================
print(f"\n{'='*20} MS-MARCO {'='*20}")
seed_metrics = []

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    metrics, weights = run_single_experiment(train_data, test_data, seed)
    seed_metrics.append(metrics)

    print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
    print(f"|--------|-----|-----|-----|-----|----------|")
    for method, m in metrics.items():
        print(f"| {method} | {m['R@1']:.3f} | {m['R@3']:.3f} | {m['R@5']:.3f} | {m['MRR']:.3f} | {m['Avg Rank']:.2f} |")

    print("\nWeights:")
    for name, w in weights.items():
        print(f"  {name} {np.round(w, 3)}")

# average across seeds
print(f"\n=== AVERAGE RESULTS (MS-MARCO) ===")
print(f"\n| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |")
print(f"|--------|-----|-----|-----|-----|----------|")
methods = list(seed_metrics[0].keys())
for method in methods:
    avg = {k: np.mean([s[method][k] for s in seed_metrics]) for k in ["R@1", "R@3", "R@5", "MRR", "Avg Rank"]}
    print(f"| {method} | {avg['R@1']:.3f} | {avg['R@3']:.3f} | {avg['R@5']:.3f} | {avg['MRR']:.3f} | {avg['Avg Rank']:.2f} |")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]


==================== MS-MARCO ====================

--- Seed 42 ---
Built 656/800 samples
Built 186/200 samples


Test (seed=42): 100%|██████████| 186/186 [00:16<00:00, 10.99it/s]


| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.930 | 1.000 | 1.000 | 0.965 | 1.07 |
| CE | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| NLI | 0.898 | 1.000 | 1.000 | 0.949 | 1.10 |
| Physarum | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| CE+NLI | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| CE+NLI+Sim | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| CE+Physarum | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| NLI+Physarum | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| CE+NLI+Physarum | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |
| CE+NLI+Physarum+Sim | 0.957 | 1.000 | 1.000 | 0.978 | 1.04 |

Weights:
  CE+NLI [3.399 1.219]
  CE+NLI+Sim [ 3.944  1.242 -0.644]
  CE+Physarum [0.809 3.452]
  NLI+Physarum [1.138 3.524]
  CE+NLI+Physarum [0.536 1.125 3.014]
  CE+NLI+Physarum+Sim [ 1.08   1.157  3.261 -0.911]

=== AVERAGE RESULTS (MS-MARCO) ===

| Method | R@1 | R@3 | R@5 | MRR | Avg Rank |
|--------|-----|-----|-----|-----|----------|
| Sim | 0.930 | 1.000 | 1.000 | 0